In [69]:
import pandas as pd
from sklearn.metrics import classification_report

In [70]:
# Experiment we want to explore - customise these
TARGET_LABEL = "object_region"
EXPERIMENT_NUMBER = 1
FOLDER_NAME = f"{TARGET_LABEL}_classify"
RESULTS_PATH = f"../results/{FOLDER_NAME}"
EXPERIMENTS_DF_PATH = f"{RESULTS_PATH}/experiments.parquet"

In [71]:
# Load the required experiment results from the Parquet file
df = pd.read_parquet(EXPERIMENTS_DF_PATH)
df.head()

,experiment_number,experiment_name,model_config,data_config,model_type,list_classes,history,test_acc,test_loss,weighted_f1_avg,y_pred,y_true
0,1,cnn_adamw_10_v2,{'checkpoint_dir': 'checkpoints/object_region_...,"{'batch_size': 128, 'bg_path': 'data/baseline....",baseline,"[beans_base, beans_body, beans_lid, hammer_han...","[{'epoch': 1, 'train_acc': 89.66611479028697, ...",78.115385,1.732033,0.778507,"[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,1,cnn_adamw_10_v2,{'checkpoint_dir': 'checkpoints/object_region_...,"{'batch_size': 128, 'bg_path': 'data/baseline....",resnet18,"[beans_base, beans_body, beans_lid, hammer_han...","[{'epoch': 1, 'train_acc': 96.4873068432671, '...",90.935897,0.259292,0.909929,"[9, 9, 9, 9, 9, 7, 9, 7, 9, 9, 7, 9, 9, 9, 9, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [72]:
def print_experiment_info(experiment_number):
    """
    Print the model and data configurations, training history, and evaluation results for a
    given experiment number.

    Args:
        experiment_number (int): The experiment number to display information for.
    """
    print(f"Experiment Number: {experiment_number}\n")

    # Filter DataFrame for the specified experiment number
    df_experiment = df[df["experiment_number"] == experiment_number]
    if df_experiment.empty:
        raise ValueError(f"Experiment number {experiment_number} not found in DataFrame.")

    # Print the model and data configurations (from first row - same for all)
    print("Model Config:")
    for key, value in df_experiment.iloc[0]["model_config"].items():
        print(f"  {key}: {value}")
    print("\nData Config:")
    for key, value in df_experiment.iloc[0]["data_config"].items():
        print(f"  {key}: {value}")

    # For each row in the experiment
    for _, row in df_experiment.iterrows():
        print(f"\n---- Model {row['model_type']} ----\n")

        # Print training history
        history = row["history"]
        print("Training History:")
        for epoch_idx, metrics in enumerate(history):
            print(f"  Epoch {epoch_idx + 1}:")
            for metric_name, metric_value in metrics.items():
                print(f"    {metric_name}: {metric_value}")

        # Print evaluation results
        print("\nEvaluation Results:")
        print(f"  Test Accuracy: {row['test_acc']:.4f}")
        print(f"  Test Loss: {row['test_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['weighted_f1_avg']:.4f}")

In [73]:
def print_classification_reports(experiment_number):
    """
    Print the classification reports for a given experiment number.

    Args:
        experiment_number (int): The experiment number to display the reports for.
    """
    print(f"Classification Reports for Experiment Number: {experiment_number}\n")

    # Filter DataFrame for the specified experiment number
    df_experiment = df[df["experiment_number"] == experiment_number]
    if df_experiment.empty:
        raise ValueError(f"Experiment number {experiment_number} not found in DataFrame.")

    # For each row in the experiment
    for _, row in df_experiment.iterrows():
        print(f"\n---- Model {row['model_type']} ----\n")
        report = classification_report(
            row["y_true"], row["y_pred"], target_names=row["list_classes"]
        )
        print(report)

In [74]:
print_experiment_info(EXPERIMENT_NUMBER)

Experiment Number: 1

Model Config:
  checkpoint_dir: checkpoints/object_region_classify/001
  learning_rate: 0.0004
  model_title: baseline
  momentum: 0.9
  num_epochs: 10
  optimizer: adamw
  weight_decay: 0.04

Data Config:
  batch_size: 128
  bg_path: data/baseline.jpg
  norm_cache_path: configs/norm_cache.json
  norm_type: dataset
  num_workers: 4
  random_state: 146
  shuffle_map: {'test': False, 'train': True, 'val': False}
  split_size: 0.3
  stratify_label: object_region
  transform_name: pad_224
  unseen_objs: None

---- Model baseline ----

Training History:
  Epoch 1:
    epoch: 1
    train_acc: 89.66611479028697
    train_loss: 0.3380541432645618
    val_acc: 71.6923076923077
    val_loss: 1.379814376030809
  Epoch 2:
    epoch: 2
    train_acc: 99.24668874172185
    train_loss: 0.03225715252124822
    val_acc: 76.12820512820512
    val_loss: 1.3284726301410017
  Epoch 3:
    epoch: 3
    train_acc: 99.57505518763797
    train_loss: 0.014973550141931668
    val_acc: 75.69

In [75]:
print_classification_reports(EXPERIMENT_NUMBER)

Classification Reports for Experiment Number: 1


---- Model baseline ----

                  precision    recall  f1-score   support

      beans_base       0.71      0.76      0.74       480
      beans_body       0.74      0.78      0.76       480
       beans_lid       0.97      0.42      0.59       480
   hammer_handle       0.68      0.76      0.71       480
     hammer_head       0.53      0.90      0.67       360
     hammer_neck       0.81      0.51      0.63       480
   pringles_base       0.98      0.96      0.97       360
   pringles_body       0.81      0.61      0.70       360
    pringles_lid       0.80      0.93      0.86       360
 scissors_blades       0.66      0.80      0.72       600
 scissors_handle       0.96      0.74      0.84       720
tennis_ball_body       0.82      0.93      0.87       600
tennis_ball_seam       1.00      0.97      0.98       720
toilet_roll_body       0.84      0.70      0.77       600
toilet_roll_ends       0.68      0.87      0.76      